In [ ]:
# -*- coding: utf-8 -*-

Tarea1_JavieraManzo.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1joKYLlXOEttfVw0_UnYMuWLNQtKnpiCS

# Tarea Nº 1 - Inteligencia Artificial
### Alumno: Javier Manzo González
### Fecha de entrega: 13 de Septiembre 2024

## Introducción

En la presente tarea se desarrollan tres actividades aplicando técnicas de inteligencia artificial, utilizando redes bayesianas y modelos de cadenas de Markov.
El primer ítem consiste en escoger un dataset con el fin de aprender la estructura de una red bayesiana usando métodos de búsqueda como Exhaustive search e Hillclimb search, para luego realizar el aprendizaje de parámetros e inferencias.
Para el siguiente ítem se nos otorga una matriz de probabilidades de transición de badas para la cual se debe implementar una cadena de Markov y realizar un random-walk, además de calcular la distribución estacionaria de la cadena.
Por último se debe utilizar el modelo oculto de markov (HMM) visto en clases para determinar el estado oculto q5 más probable en una secuencia de observaciones.

##librerías necesarias
Las librerias necesarias para la realizacion de toda la tarea son las siguientes:

—bnlearn: librería de Python diseñada para trabajar con redes bayesianas. Esta nos permite estructurar redes a partir de datos, y realizar inferencias y llevar a cabo aprendizaje de parámetros.

—NumPy: librería de Python para el cálculo numérico. Utilizada para el trabajo con matrices y grandes arreglos, cargada con funciones para realizar operaciones matemáticas y estadísticas de manera eficiente.

—Matplotlib: librería de Python diseñada para la creación de gráficos y visualización de datos.

—hmmlearn: librería de Python creada para la implementación de modelos ocultos de Markov (HMM).

In [ ]:
#instalacion
!pip install bnlearn hmmlearn numpy matplotlib

#import
import bnlearn as bn
import numpy as np
import matplotlib.pyplot as plt
from hmmlearn import hmm

#1.Red Bayesiana
Para el desarrollo del primer ítem se seleccionó un dataset extraído de Kaggle, llamada “depression data”, esta base de datos presenta más de 400.000 filas y 16 columnas en las cuales se enlista personas y sus correspondientes datos los cuales corresponden a aspectos importantes sobre factores personales y estilos de vida de personas con depresión, los parámetros establecidos son nombre, edad, estado civil, nivel educacional, número de hijos, si es fumador, el nivel de actividad física, si tiene trabajo o no, los ingresos, el consumo de alcohol, hábitos alimenticios, calidad del sueño, antecedentes de enfermedad mental, antecedentes de abuso de sustancias, historial familiar y si presenta alguna enfermedad crónica.

Utilizando el tutorial proporcionado por la ayudante se exporta el dataset desde Kaggle, además se limitó el número de filas a 10.000, ya que el entorno no permitía el total de filas y el tiempo de procesamiento podía tardar días:

In [ ]:
from google.colab import files
uploaded = files.upload()

!pip install -q kaggle

! mkdir ~/.kaggle
! cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d teejmahal20/airline-passenger-satisfaction

!unzip /content/depression-dataset.zip #extraccion del archivo .csv

##Hillclimb Search:
Se comienza con el método de búsqueda Hillclimb, este método como su nombre lo dice se basa en la búsqueda en pendiente, esto quiere decir que se comienza con una solución inicial y se realizan mejoras iterativas para llegar a una mejor solución. Esto no garantiza encontrar la mejor solución global, pero presenta un proceso más eficiente que la búsqueda exhaustiva.

In [ ]:
import pandas as pd
df = pd.read_csv('depression_data.csv', on_bad_lines='skip',nrows=10000)
print(df.shape)
df.head()

# Usando HillClimb Search
H= bn.structure_learning.fit(df, methodtype='hillclimbsearch')
G = bn.plot(H, interactive=True, params_interactive={'notebook':True})

# Aprender los parámetros
params_H = bn.parameter_learning.fit(H, df)
bn.plot(params_H)

###Inferencias de Hillclimb:
1.¿Cúal es la probabilidad de que una persona con depresión este casada y desepleada?

In [ ]:
q_1 = bn.inference.fit(params_H, variables=['Marital Status'], evidence={'Employment Status': 'Unemployed'})

2.¿Cúal es la probabilidad de que las personas jovenes (menores de 25 años) con depresión sean fumadores?

In [ ]:
q_2 = bn.inference.fit(params_H, variables=['Age'], evidence={'Smoking Status': 'Current'})

##Exhaustive Search:
La búsqueda exhaustiva también se conoce como búsqueda de fuerza bruta, ya que se prueban todas las posibles soluciones para encontrar la óptima. A pesar de que es la técnica más sencilla y garantiza encontrar la mejor opción, es muy ineficiente y tarda considerablemente más tiempo que Hillclimb Serch.
En este caso aunque se disminuyo la cantidad de filas(de 10.000 a 50) y la cantidad de columnas (de 16 a 6) no se logró obtener resultados, ya que aún después de 1 hora de ejecución no terminaba el proceso.

In [ ]:
de = pd.read_csv('depression_data.csv', on_bad_lines='skip',nrows=50,usecols=['Name','Age', 'Marital Status', 'Smoking Status','Education Level','Number of Children','Smoking Status'])
print(de.shape)
de.head()

# Usando # Search
E= bn.structure_learning.fit(de, methodtype='exhaustivesearch')
G = bn.plot(E, interactive=True, params_interactive={'notebook':True})

# Aprender los parámetros
params_E = bn.parameter_learning.fit(E, de)
bn.plot(params_E)

#2. Cadena de Markov
La cadena de Markov es un modelo matemático que describe un sistema que cambia de estado de acuerdo con ciertas probabilidades. Los estados del sistema están representados por un conjunto finito y las transiciones entre estados son determinadas por una matriz de transición.
La distribución estacionaria de una cadena de Markov es un vector de probabilidad que describe la proporción de tiempo que el sistema pasa en cada estado a lo largo del plazo una vez se llega al equilibrio.
Matemáticamente, se llega a la distribución estacionaria desde un vector inicial π multiplicado por la matriz B tantas veces sea necesario hasta que los valores no presenten más cambios y se alcance el estado estable.
Para el siguiente ítem se nos entrega una matriz la cual representa la probabilidad de transición de nueve bandas:

In [ ]:
B = np.array([[0.25,0.06,0.08,0.15,0.04,0.02,0.15,0.15,0.10],
                [0.15,0.15,0.10,0.22,0.01,0.02,0.15,0.10,0.10],
                [0.12,0.00,0.05,0.24,0.14,0.04,0.27,0.07,0.07],
                [0.05,0.13,0.05,0.30,0.10,0.10,0.22,0.05,0.00],
                [0.18,0.20,0.07,0.20,0.15,0.05,0.05,0.05,0.05],
                [0.20,0.10,0.20,0.05,0.05,0.10,0.02,0.15,0.13],
                [0.01,0.05,0.15,0.14,0.17,0.10,0.12,0.10,0.16],
                [0.17,0.15,0.07,0.07,0.15,0.10,0.12,0.09,0.08],
                [0.13,0.11,0.13,0.03,0.20,0.20,0.04,0.15,0.01]])
states = np.array(['Helloween', 'Hammerfall', 'Stratovarius', 'Rhapsody of fire', 'Yngwie Malmsteen', 'Liquid Tension Experiment', 'Blind Guardian','Dream Theater','Symphony X'])

Una vez definida la matriz y los estados, se define el vector de probabilidad de inicio basándonos en el enunciado que nos dice que se empieza escuchando a “Yngwie Malmsteen” que es g5:

In [ ]:
init = np.array([0, 0, 0, 0, 1, 0, 0, 0, 0])

Multiplicando las matrices de transición con el vector de probabilidad de inicio obtenemos la probabilidad de que la siguiente canción que se escuche sea de cada una de las bandas de g1 a g9 respectivamente.

In [ ]:
next = init @ B
next

Para poder determinar el estado estacionario se multiplica múltiples veces la matriz con el vector de probabilidad, esto en un ciclo donde el vector va tomando el valor del resultado, se comienza con un valor grande y se observa en la gráfica desde que valor comienza la distribución estacionaria:

In [ ]:
T_next100 = [init]
for i in range(100):
    T_next100.append(T_next100[-1] @ B)
T_100 = T_next100[100]
print(T_100)

print("Banda mas probable de escuchar mas tiempo: ",states[T_next100[100].argmax()])
T_next100_distributions = pd.DataFrame(T_next100)
print(T_next100_distributions)
T_next100_distributions.plot()

En el gráfico podemos apreciar que se presenta una distribución estacionaria desde valores mucho menores por lo que se disminuye el valor a 20:

In [ ]:
T_next20 = [init]
for i in range(20):
    T_next20.append(T_next20[-1] @ B)
T_20 = T_next20[20]
print(T_20)

print("Banda mas probable de escuchar mas tiempo: ",states[T_next20[20].argmax()])
T_next20_distributions = pd.DataFrame(T_next20)
print(T_next20_distributions)
T_next20_distributions.plot()

En este segundo gráfico y tabla podemos apreciar que la distribución estacionaria se logra en la iteración número 11 y que el estado con mayor valor es “Rhapsody of fire” (g4).

# 3. Modelo oculto de markov (HMM)
Este modelo es una extensión de las cadenas de Markov y se utiliza para modelar sistemas donde los estados internos de este no son observables, pero sí se tienen observaciones influenciadas por estos estados ocultos. En este caso los estados nos fueron proporcionados por el profesor durante la clase y son “comer ramen”(R), “comer Salmorejo”(R) y “comer sopa de cebolla”(C) siendo las observaciones “satisfecho”(S) e “insatisfecho”(I), también se nos proporcionan las probabilidades de cada uno y el vector inicial el que indica que se inicia con salmorejo.

In [ ]:
# Definir los estados y observaciones
Estados = ['ramen', 'salmorejo', 'sopa_cebolla']
Observaciones = ['satisfecho', 'insatisfecho']

# Convertir observaciones a índices
obs_map = {obs: i for i, obs in enumerate(Observaciones)}

# Secuencia de observaciones entregadas por el enunciado
secuencia = ['satisfecho', 'insatisfecho', 'insatisfecho', 'satisfecho', 'insatisfecho', 'insatisfecho', 'insatisfecho', 'insatisfecho', 'satisfecho']
obs_indices = [obs_map[obs] for obs in secuencia]

# Definir el modelo HMM
model = hmm.MultinomialHMM(n_components=3,init_params='e')

#traspasar los datos otorgados por el profesor en la clase
#vector inicial
model.startprob_ = np.array([0, 1, 0])
#Matriz de probabilidad de estados(R,S,C)
model.transmat_ = np.array([[0.2, 0.6, 0.2],
                            [0.3, 0, 0.7],
                            [0.5, 0, 0.5]])
#Matriz de probabilidad de observaciones segun estados (S,I)
model.emissionprob_ = np.array([[0.8, 0.2],
                                [0.3, 0.7],
                                [0.6, 0.4]])

# Ajustar el modelo a la secuencia de observaciones
model.fit(np.array(obs_indices).reshape(-1, 1))

# Predecir la secuencia de estados más probable y extaer el q5
states_predicted = model.predict(np.array(obs_indices).reshape(-1, 1))
estado_q5 = states_predicted[4]

# Imprimir el estado más probable para q5
print(f'Estado más probable para q5: {Estados[estado_q5]}')

Dado el patrón de observaciones y las matrices de probabilidad se obtuvo que el estado oculto para el tiempo q5 era comer sopa de cebolla y se produjo la observación de insatisfecho. Esto refleja como el modelo está usado la secuencia previa de observaciones para inferir el tipo de comida en un momento específico.